# USGS Water Data Demo

This notebook demonstrates how modern Python retrieval calls can replace the legacy NWIS/.NET workflow for station discovery and time-series retrieval.

In [6]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.usgs_waterdata_demo import DEMO_CONFIG, LEGACY_CALLS, NOTEBOOK_MAPPING_ROWS, get_usgs_stations_in_bbox, get_usgs_timeseries


## Demo Configuration

The demo uses explicit configuration so the station discovery, retrieval, and export examples stay deterministic across runs.

In [7]:
DEMO_CONFIG

{'bbox': (-82.859, 31.899, -82.013, 32.947),
 'output_dir': 'outputs/usgs_waterdata_demo',
 'date_range': ('1880-01-01', '2100-01-01'),
 'example_stations': {'discharge': {'station_id': '02225500',
   'parameter_code': '00060'},
  'gw_daily': {'station_id': '324209082430201', 'parameter_code': '72019'},
  'gw_periodic': {'station_id': '410947071344803', 'parameter_code': '72019'}}}

## Legacy to Modern Mapping

The table below shows how each legacy workflow step maps to a modern helper entry point and the underlying `dataretrieval.waterdata` calls used in this demo.

In [9]:
mapping_frame = pd.DataFrame(NOTEBOOK_MAPPING_ROWS)
display(mapping_frame)

,legacy_workflow_step,workflow_type,legacy_intent,modern_helper_entry_point,waterdata_calls,important_filters,expected_output_shape
0,New Project station discovery,discharge,Find surface-water stations in a bounding box,get_usgs_stations_in_bbox,get_monitoring_locations + get_time_series_met...,"site_type_code=ST, parameter_code=00060",DataFrame of discharge-capable stations
1,New Project station discovery,gw_daily,Find groundwater stations with daily series in...,get_usgs_stations_in_bbox,get_monitoring_locations + get_time_series_met...,site_type_code=GW,DataFrame of groundwater daily stations
2,New Project station discovery,gw_periodic,Find groundwater stations with periodic observ...,get_usgs_stations_in_bbox,get_monitoring_locations + get_field_measurements,site_type_code=GW,DataFrame of groundwater periodic stations
3,Adding Data to Project,discharge,Retrieve discharge time series for a selected ...,get_usgs_timeseries,get_daily,"parameter_code=00060, configured date_range",normalized DataFrame + records + files
4,Adding Data to Project,gw_daily,Retrieve groundwater daily time series for a s...,get_usgs_timeseries,get_daily,"configured parameter_code, configured date_range",normalized DataFrame + records + files
5,Adding Data to Project,gw_periodic,Retrieve groundwater periodic measurements for...,get_usgs_timeseries,get_field_measurements,"configured parameter_code, configured date_range",normalized DataFrame + records + files


## Station Discovery

This section runs the bounding-box station discovery workflow for `discharge`, `gw_daily`, and `gw_periodic`, then shows row counts and sample records for each workflow type.

In [10]:
station_results = get_usgs_stations_in_bbox(DEMO_CONFIG["bbox"], ["discharge", "gw_daily", "gw_periodic"])
for workflow_type, frame in station_results.items():
    print(f"{workflow_type}: {len(frame)} stations")
    if frame.empty:
        print("No stations returned for this workflow type")
    else:
        display(frame.head())


TypeError: get_monitoring_locations() got an unexpected keyword argument 'bBox'

## Time-Series Retrieval and Exports

This section uses the deterministic example stations from the config to retrieve time-series data, normalize the result, and write JSON, CSV, and Parquet outputs when supported.

In [11]:
timeseries_results = {}
for workflow_type, station_config in DEMO_CONFIG["example_stations"].items():
    timeseries_results[workflow_type] = get_usgs_timeseries(
        station_config["station_id"],
        workflow_type,
        DEMO_CONFIG["output_dir"],
        parameter_code=station_config["parameter_code"],
        date_range=DEMO_CONFIG["date_range"],
    )
    print(workflow_type, timeseries_results[workflow_type]["files"])
    if timeseries_results[workflow_type]["frame"].empty:
        print(timeseries_results[workflow_type]["metadata"].get("warning"))
    else:
        display(timeseries_results[workflow_type]["frame"].head())


TypeError: get_daily() got an unexpected keyword argument 'start'

## Notes and Limits

- This demo favors readability over framework-level abstraction.
- Shapefile and projection workflows are intentionally out of scope.
- The helper module can be reused for follow-on migration work if the notebook proves the workflow.